In [ ]:
import os
import time
import json
import random
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

DATA_DIR = '../数据'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(service=service, options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
driver.set_page_load_timeout(30)

In [ ]:
city_codes = {
    '北京': '010000', '上海': '020000', '广州': '030200', '深圳': '040000',
    '杭州': '080200', '成都': '090200', '武汉': '180200', '南京': '070200'
}

In [ ]:
def parse_card(card):
    try:
        data = json.loads(card.get_attribute('sensorsdata'))
        company = card.find_element(By.CSS_SELECTOR, 'a.comp .cname').get_attribute('title')
        dc_list = card.find_elements(By.CSS_SELECTOR, 'a.comp .bc span.dc')
        industry = dc_list[0].text if len(dc_list) > 0 else ''
        company_type = dc_list[1].text if len(dc_list) > 1 else ''
        company_size = dc_list[2].text if len(dc_list) > 2 else ''
        tags = card.find_elements(By.CSS_SELECTOR, '.joblist-item-tags .tags > div')
        tag_text = ','.join(t.get_attribute('title') for t in tags)
        return {
            '岗位名称': data.get('jobTitle'), '薪资范围': data.get('jobSalary'),
            '工作城市': data.get('jobArea'), '学历要求': data.get('jobDegree'),
            '工作经验': data.get('jobYear'), '发布时间': data.get('jobTime'),
            '公司名称': company, '行业类别': industry, '公司性质': company_type,
            '公司规模': company_size, '技能标签': tag_text
        }
    except:
        return None

In [ ]:
def scrape_one_search(city, keyword, pages=5):
    city_code = city_codes[city]
    url = f'https://we.51job.com/pc/search?jobArea={city_code}&keyword={quote(keyword)}&searchType=2&keywordType='

    for attempt in range(3):
        try:
            driver.get(url)
            time.sleep(random.uniform(4, 6))
            all_records = []

            for page in range(pages):
                cards = driver.find_elements(By.CSS_SELECTOR, '.joblist-item-job')
                if not cards and page == 0:
                    raise ValueError("Blocked or No Data")

                for c in cards:
                    res = parse_card(c)
                    if res:
                        all_records.append(res)

                if page < pages - 1:
                    next_btn = driver.find_elements(By.CSS_SELECTOR, 'button.btn-next')
                    if not next_btn or not next_btn[0].is_enabled():
                        break
                    driver.execute_script("arguments[0].click();", next_btn[0])
                    time.sleep(random.uniform(3, 5))
            return all_records
        except Exception:
            time.sleep(random.uniform(10, 15))
            driver.refresh()
    return []

In [56]:
keywords = ['数据分析', 'Python', '机器学习', '数据挖掘']
cities = list(city_codes.keys())
all_data = []

for city in cities:
    for keyword in keywords:
        records = scrape_one_search(city, keyword, pages=5)
        all_data.extend(records)
        print(f"{city} {keyword} {len(records)}")
        time.sleep(random.uniform(4, 8))

北京 数据分析 100
北京 Python 100
北京 机器学习 100
北京 数据挖掘 100
上海 数据分析 100
上海 Python 100
上海 机器学习 100
上海 数据挖掘 100
广州 数据分析 100
广州 Python 100
广州 机器学习 100
广州 数据挖掘 100
深圳 数据分析 100
深圳 Python 100
深圳 机器学习 100
深圳 数据挖掘 100
杭州 数据分析 100
杭州 Python 100
杭州 机器学习 100
杭州 数据挖掘 100
成都 数据分析 100
成都 Python 100
成都 机器学习 100
成都 数据挖掘 100
武汉 数据分析 100
武汉 Python 100
武汉 机器学习 100
武汉 数据挖掘 100
南京 数据分析 100
南京 Python 100
南京 机器学习 100
南京 数据挖掘 100


In [57]:
len(all_data)

3200

In [58]:
import pandas as pd
df = pd.DataFrame(all_data)
df.to_csv(os.path.join(DATA_DIR, 'raw_jobs.csv'), index=False, encoding='utf-8-sig')

In [59]:
df.shape

(3200, 11)

In [60]:
df.head()

,岗位名称,薪资范围,工作城市,学历要求,工作经验,发布时间,公司名称,行业类别,公司性质,公司规模,技能标签
0,财务助理与数据分析员（外包岗位）,14-20万/年,北京·海淀区,本科,无需经验,2026-05-29 16:31:32,北京纵横机电科技有限公司,交通/运输/客运,国企,1000-5000人,"五险一金,定期体检,免费班车,意外医疗保险,各类法定休假,日常健身,出差补助,有竞争的薪酬,工作餐"
1,医药数据分析岗,8千-1.2万,北京·朝阳区,本科,1年及以上,2026-02-05 09:24:29,华润双鹤药业股份有限公司,制药/生物工程丨化工,已上市,10000人以上,"数据库,医药行业,处方药,销售指标,数据指标体系,评估市场容量,excel,数据管理,客户管..."
2,数据分析策略专员,1.2-2万,北京·丰台区,本科,3年及以上,2026-05-26 14:57:33,新华网股份有限公司,互联网/电子商务,国企,1000-5000人,"计算机,数据分析,excel,sql,市场营销,数学,报表,互联网,ai,工商管理"
3,数据分析工程师（北京）,8千-1.2万·13薪,北京·大兴区,本科,1-3年,2026-04-16 09:20:43,中科仪（广州）半导体设备有限责任公司,机械/设备/重工,国企,50-150人,"故障诊断,数据分析,半导体,数据清洗,算法维护,统计学,寿命预测,相关性分析,五险一金,交通..."
4,SFE数据分析专员/主管(J11947),7千-1万,北京·东城区,本科,1年及以上,2026-03-18 14:08:21,贝达药业股份有限公司,制药/生物工程,合资,1000-5000人,"管理规范,培训"
